# Tubes 2 IF3270 — CNN, RNN & LSTM
**Dataset CNN**: Intel Image Classification (~25.000 gambar, 6 kelas)
**Dataset RNN/LSTM**: Flickr8k (image captioning)


In [ ]:
# =============================================================================
# GPU Setup — jalankan PERTAMA sebelum import TF lain
# =============================================================================
import tensorflow as tf

gpus = tf.config.list_physical_devices('GPU')
if gpus:
    for gpu in gpus:
        tf.config.experimental.set_memory_growth(gpu, True)
    print(f'[GPU] Ditemukan {len(gpus)} GPU: {[g.name for g in gpus]}')
    print(f'[GPU] TensorFlow versi: {tf.__version__}')
else:
    print('[GPU] Tidak ada GPU terdeteksi — menggunakan CPU.')
    print('      Untuk GPU NVIDIA di Windows, install: pip install tensorflow==2.10.0')
    print('      Atau gunakan WSL dan: pip install tensorflow[and-cuda]')
    print(f'[TF] Versi TensorFlow: {tf.__version__}')

print(f'[TF] Built with CUDA: {tf.test.is_built_with_cuda()}')

In [ ]:
import os
import sys
import numpy as np

# Setup path agar modul src/ dapat diimport
SRC_DIR = os.path.abspath('.')  # asumsi notebook dijalankan dari folder src/
PROJECT_ROOT = os.path.dirname(SRC_DIR)

# Jika notebook dijalankan dari root project, sesuaikan:
if not os.path.exists(os.path.join(SRC_DIR, 'cnn')):
    SRC_DIR = os.path.join(os.path.abspath('.'), 'src')
    PROJECT_ROOT = os.path.abspath('.')

sys.path.insert(0, SRC_DIR)
sys.path.insert(0, os.path.join(SRC_DIR, 'shared'))

print(f'[Path] SRC_DIR: {SRC_DIR}')
print(f'[Path] PROJECT_ROOT: {PROJECT_ROOT}')

DATA_DIR = os.path.join(PROJECT_ROOT, 'data', 'intel_image_classification')
WEIGHTS_DIR = os.path.join(PROJECT_ROOT, 'weights', 'cnn')
RESULTS_DIR = os.path.join(PROJECT_ROOT, 'results')

print(f'[Path] DATA_DIR: {DATA_DIR} (exists: {os.path.exists(DATA_DIR)})')

## Bagian 1: Utility Functions (PIL/Pillow + NumPy)

In [ ]:
from shared.preprocessing import load_image, load_batch, extract_features

# Test load_image
# img = load_image('path/ke/gambar.jpg', target_size=(150, 150))
# print(f'Image shape: {img.shape}, dtype: {img.dtype}, range: [{img.min():.2f}, {img.max():.2f}]')

# Test load_batch
# batch = load_batch(['path1.jpg', 'path2.jpg'], target_size=(150, 150))
# print(f'Batch shape: {batch.shape}')  # (N, 150, 150, 3)

## Bagian 2: Forward Propagation From Scratch

In [ ]:
from cnn.scratch.conv2d import Conv2D
from cnn.scratch.locally_connected2d import LocallyConnected2D
from cnn.scratch.pooling import MaxPooling2D, AveragePooling2D, GlobalAveragePooling2D
from cnn.scratch.flatten import Flatten
from cnn.scratch.model_scratch import CNNScratch

## Bagian 3: Pelatihan Model (Keras)

Variasi hyperparameter (16 arsitektur):
- Jumlah layer konvolusi: [2, 4]
- Jumlah filter: [32, 128]
- Ukuran kernel: [(3,3), (5,5)]
- Pooling: ['max', 'average']

Total: 2 × 2 × 2 × 2 = **16 arsitektur**

In [ ]:
from shared.intel_preprocess import IntelImagePreprocessor

preprocessor = IntelImagePreprocessor(DATA_DIR, target_size=(150, 150))
preprocessor.load_data()
preprocessor.summary()

In [ ]:
from cnn.keras.train import train_with_variations

results = train_with_variations(
    data_dir=DATA_DIR,
    arch_type='conv2d',
    layer_variations=[2, 4],
    filter_variations=[32, 128],
    kernel_variations=[(3, 3), (5, 5)],
    pooling_variations=['max', 'average'],
    epochs=30,
    batch_size=32,
    weights_dir=WEIGHTS_DIR,
    results_path=os.path.join(RESULTS_DIR, 'cnn_variations.json')
)

## Bagian 4: Eksperimen dan Evaluasi

In [ ]:
from cnn.keras.evaluate import run_part4_evaluation

# Evaluasi dengan best model dari Bagian 3
# run_part4_evaluation(...)